In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml kailash-kaizen

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# M6 Ollama bootstrap — local LLMs, no API keys
# On Colab: installs + starts Ollama and pulls this lesson's models.
# Locally: verifies the daemon is running and models are pulled.
# ══════════════════════════════════════════════════════════════════
_LESSON_MODELS = ['qwen2.5:0.5b', 'llama3.2:3b']
import sys, os, time, shutil, subprocess
import httpx

_OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
_IN_COLAB = 'google.colab' in sys.modules

def _daemon_up(timeout_s=1.0):
    try:
        r = httpx.get(f'{_OLLAMA_BASE_URL}/api/tags', timeout=timeout_s)
        r.raise_for_status()
        return r.json()
    except Exception:
        return None

if _IN_COLAB and shutil.which('ollama') is None:
    print('[ollama] installing binary …')
    subprocess.run(
        'curl -fsSL https://ollama.com/install.sh | sh',
        shell=True, check=True,
    )

if _IN_COLAB and _daemon_up() is None:
    print('[ollama] starting daemon …')
    subprocess.Popen(
        'nohup ollama serve > /tmp/ollama.log 2>&1 &',
        shell=True,
    )
    deadline = time.monotonic() + 30
    while time.monotonic() < deadline and _daemon_up() is None:
        time.sleep(0.5)

_tags = _daemon_up(timeout_s=2.0)
if _tags is None:
    raise RuntimeError(
        f'Ollama daemon not reachable at {_OLLAMA_BASE_URL}. '
        'On Colab: re-run this cell. Locally: run `ollama serve`.'
    )

_have = {m.get('name','').split(':',1)[0] for m in _tags.get('models', [])}
for _model in _LESSON_MODELS:
    _fam = _model.split(':', 1)[0]
    if _fam in _have:
        continue
    if _IN_COLAB:
        print(f'[ollama] pulling {_model} (one-time) …')
        subprocess.run(['ollama', 'pull', _model], check=True)
    else:
        raise RuntimeError(
            f'Model {_model!r} not pulled. Run: ollama pull {_model}'
        )

print(
    '✓ Ollama ready at', _OLLAMA_BASE_URL,
    '— models:', _LESSON_MODELS,
)



In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp06
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp06/_ollama_bootstrap.py ──
"""Ollama bootstrap for MLFP06 — local LLMs, no API keys, real calls only.

Why this module exists
----------------------
M6 trains, prompts, evaluates, and deploys real LLMs. Routing every call
through OpenAI:

  * forces every learner to hold an API key
  * burns budget every time a notebook reruns a cell
  * makes "did the LLM actually fire?" invisible (a stubbed `cost=$0`
    looks identical to a successful call)

This module switches every M6 LLM call to a locally-running Ollama daemon
that the student installs once and re-uses across every lesson.

Contract
--------
* :func:`make_delegate` — the only sanctioned way to construct a Kaizen
  ``Delegate`` for M6. Wraps the Ollama adapter with sensible defaults and
  forces ``budget_usd=None`` (Kaizen mis-prices Ollama as $3/$15-per-Mtok;
  budget tracking against a free local provider is meaningless).
* :func:`make_embedder` — the only sanctioned way to construct an embedding
  adapter for M6 RAG (lesson 6.4).
* :func:`preflight_ollama` — pre-flight reachability + model-presence check.
  Raises :class:`OllamaUnreachableError` with an actionable message. Call
  this at the top of every notebook / script before constructing a
  Delegate.
* :data:`LESSON_MODELS` — manifest of which models each lesson needs.
  Consumed by ``scripts/generate_selfcontained_notebook.py`` to inject the
  Colab bootstrap cell.

Per ``rules/zero-tolerance.md`` Rule 2: there is no silent stub fallback
in this module. If Ollama is not reachable, we raise. The previous
``run_delegate(...) -> ("unknown", 0.0, latency)`` fallback is BLOCKED.

Per ``rules/framework-first.md``: every LLM call goes through Kaizen
``Delegate``. Raw ``httpx`` calls to ``/api/generate`` are BLOCKED for
generation paths; ``httpx`` is used here only for the daemon-reachability
preflight.
"""

import logging
import os
import shutil
import subprocess
import sys
import time
from typing import Any

import httpx
from dotenv import load_dotenv

logger = logging.getLogger(__name__)

# ════════════════════════════════════════════════════════════════════════
# Configuration — env-driven per rules/env-models.md
# ════════════════════════════════════════════════════════════════════════

load_dotenv()


def _env(key: str, default: str) -> str:
    val = os.environ.get(key)
    return val if val else default


OLLAMA_BASE_URL: str = _env("OLLAMA_BASE_URL", "http://localhost:11434")
"""Where the Ollama daemon listens. Local default; Colab uses the same."""

DEFAULT_CHAT_MODEL: str = _env("OLLAMA_CHAT_MODEL", "llama3.2:3b")
"""Tool-capable instruct model. Allowlisted by Kaizen for function calling."""

DEFAULT_EMBED_MODEL: str = _env("OLLAMA_EMBED_MODEL", "nomic-embed-text")
"""Compact embedding model — ~270MB, 768-dim. Used by M6 RAG (lesson 6.4)."""

DEFAULT_FT_BASE_MODEL: str = _env("OLLAMA_FT_BASE_MODEL", "qwen2.5:0.5b")
"""Tiny base model for fine-tuning demos (lessons 6.2, 6.3). Fits T4."""


# ════════════════════════════════════════════════════════════════════════
# Lesson → models manifest (used by generator + preflight)
# ════════════════════════════════════════════════════════════════════════


LESSON_MODELS: dict[str, list[str]] = {
    # 6.1 Prompting — chat only
    "ex_1": [DEFAULT_CHAT_MODEL],
    # 6.2 Fine-tuning — base for LoRA + chat for evaluation
    "ex_2": [DEFAULT_FT_BASE_MODEL, DEFAULT_CHAT_MODEL],
    # 6.3 Alignment (DPO/GRPO) — base + chat for judge
    "ex_3": [DEFAULT_FT_BASE_MODEL, DEFAULT_CHAT_MODEL],
    # 6.4 RAG — chat + embeddings
    "ex_4": [DEFAULT_CHAT_MODEL, DEFAULT_EMBED_MODEL],
    # 6.5 Agents (ReAct) — tool-capable chat
    "ex_5": [DEFAULT_CHAT_MODEL],
    # 6.6 Multi-agent + MCP — tool-capable chat
    "ex_6": [DEFAULT_CHAT_MODEL],
    # 6.7 PACT governance — chat for governed Delegate
    "ex_7": [DEFAULT_CHAT_MODEL],
    # 6.8 Nexus deployment — chat behind the API
    "ex_8": [DEFAULT_CHAT_MODEL],
}


# ════════════════════════════════════════════════════════════════════════
# Errors
# ════════════════════════════════════════════════════════════════════════


class OllamaUnreachableError(RuntimeError):
    """Daemon not running, daemon unreachable, or required model not pulled.

    The error message MUST be actionable: it tells the user the exact
    command to run to fix the situation. We never wrap a low-level
    ``httpx.ConnectError`` and re-raise without translating it.
    """


# ════════════════════════════════════════════════════════════════════════
# Preflight — reachability + model presence
# ════════════════════════════════════════════════════════════════════════


def _running_in_colab() -> bool:
    return "google.colab" in sys.modules


def _list_pulled_families(*, timeout_s: float = 2.0) -> set[str]:
    """Hit /api/tags and return the set of pulled model families (no tag).

    Returns ``{"llama3.2", "qwen2.5"}`` for a daemon with
    ``llama3.2:3b`` and ``qwen2.5:0.5b`` pulled. The model family is
    everything before the first colon — we match against family rather
    than the full ``family:tag`` because pulling ``llama3.2:3b-instruct``
    or ``llama3.2:3b-instruct-q4_K_M`` should still satisfy a request
    for the ``llama3.2`` family.
    """
    url = f"{OLLAMA_BASE_URL.rstrip('/')}/api/tags"
    try:
        r = httpx.get(url, timeout=timeout_s)
        r.raise_for_status()
    except (httpx.HTTPError, httpx.ConnectError) as exc:
        raise OllamaUnreachableError(
            f"ollama daemon not reachable at {OLLAMA_BASE_URL}. "
            f"Fix: open a terminal and run `ollama serve`. "
            f"On macOS the desktop app starts the daemon automatically. "
            f"Underlying: {type(exc).__name__}: {exc}"
        ) from exc
    payload = r.json() or {}
    families: set[str] = set()
    for entry in payload.get("models", []):
        name = entry.get("name") or entry.get("model") or ""
        if not name:
            continue
        families.add(name.split(":", 1)[0])
    return families


def preflight_ollama(
    *,
    required_models: list[str] | None = None,
    auto_pull: bool | None = None,
    timeout_s: float = 2.0,
) -> None:
    """Verify the daemon is reachable and the required models are pulled.

    Args:
        required_models: Models that must be present, e.g.
            ``["llama3.2:3b", "nomic-embed-text"]``. Family-matched (the
            ``:tag`` portion is informational; any pulled tag for the
            same family satisfies the requirement).
        auto_pull: If ``True``, attempt ``ollama pull <model>`` for any
            missing model via subprocess. Defaults to ``True`` on Colab
            (where the bootstrap cell has just installed the binary)
            and ``False`` locally (where pulling is the student's job
            and surprise downloads are unwelcome).
        timeout_s: HTTP timeout for the reachability check. The default
            is intentionally short — if the daemon is alive it answers
            within 100ms.

    Raises:
        OllamaUnreachableError: Daemon unreachable, or a required model
            is missing after pull (or auto_pull was disabled).
    """
    if auto_pull is None:
        auto_pull = _running_in_colab()

    have = _list_pulled_families(timeout_s=timeout_s)
    needed = required_models or []
    missing = [m for m in needed if m.split(":", 1)[0] not in have]

    if missing and auto_pull:
        for model in missing:
            _pull_model(model)
        # Re-check
        have = _list_pulled_families(timeout_s=timeout_s)
        missing = [m for m in needed if m.split(":", 1)[0] not in have]

    if missing:
        cmds = "\n  ".join(f"ollama pull {m}" for m in missing)
        raise OllamaUnreachableError(
            f"Required model(s) not pulled: {missing}.\n"
            f"Fix — run in a terminal:\n  {cmds}"
        )

    logger.info(
        "ollama.preflight.ok",
        extra={
            "base_url": OLLAMA_BASE_URL,
            "models_present": sorted(have),
            "required": needed,
        },
    )


def _pull_model(model: str) -> None:
    """Run ``ollama pull <model>`` as a subprocess. Streams to stdout."""
    if shutil.which("ollama") is None:
        raise OllamaUnreachableError(
            f"`ollama` binary not on PATH — cannot pull {model}. "
            f"Install: https://ollama.com/download"
        )
    print(f"[ollama] pulling {model} (one-time download) …", flush=True)
    t0 = time.monotonic()
    proc = subprocess.run(
        ["ollama", "pull", model],
        capture_output=False,
        text=True,
    )
    elapsed = time.monotonic() - t0
    if proc.returncode != 0:
        raise OllamaUnreachableError(
            f"`ollama pull {model}` failed with exit {proc.returncode} "
            f"after {elapsed:.1f}s. Check network and disk space."
        )
    print(f"[ollama] {model} pulled in {elapsed:.1f}s", flush=True)


# ════════════════════════════════════════════════════════════════════════
# Delegate factory — the only sanctioned constructor for M6
# ════════════════════════════════════════════════════════════════════════


def make_delegate(
    *,
    model: str | None = None,
    tools: Any = None,
    system_prompt: str | None = None,
    temperature: float = 0.4,
    max_tokens: int = 4096,
    **kwargs: Any,
) -> Any:
    """Construct a Kaizen ``Delegate`` backed by the local Ollama daemon.

    This is the ONLY way M6 helpers and exercises construct a Delegate.
    Direct ``Delegate(...)`` construction in M6 code is BLOCKED — the
    cost-mispricing trap (``budget_usd`` against a free provider) and
    the silent OpenAI fallback that comes from omitting ``adapter=`` are
    the reasons.

    Args:
        model: Ollama model tag (e.g. ``"llama3.2:3b"``). Defaults to
            :data:`DEFAULT_CHAT_MODEL`.
        tools: Tool registry or list — passed through to Delegate.
            Auto-disables streaming on the Ollama adapter.
        system_prompt: Optional system prompt.
        temperature: Sampling temperature. Default 0.4.
        max_tokens: Hard cap on output length. Default 4096.
        **kwargs: Forwarded to ``Delegate(...)``. ``budget_usd`` is
            forced to ``None`` and ``api_key`` / ``base_url`` are
            BLOCKED (the adapter owns those).

    Returns:
        A configured Kaizen ``Delegate`` instance.
    """
    for blocked in ("api_key", "base_url", "budget_usd", "adapter"):
        if blocked in kwargs:
            raise ValueError(
                f"make_delegate(): {blocked!r} is owned by the bootstrap "
                f"and cannot be overridden by callers."
            )

    # Lazy imports — keep this module importable in environments that have
    # not yet installed kaizen-agents (e.g. early bootstrap cells).
    from kaizen_agents import Delegate
    from kaizen_agents.delegate.adapters.registry import get_adapter

    resolved = model or DEFAULT_CHAT_MODEL
    adapter = get_adapter(
        "ollama",
        model=resolved,
        base_url=OLLAMA_BASE_URL,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return Delegate(
        model=resolved,
        adapter=adapter,
        tools=tools,
        system_prompt=system_prompt,
        budget_usd=None,
        **kwargs,
    )


# ════════════════════════════════════════════════════════════════════════
# Embedding factory — the only sanctioned constructor for M6 RAG
# ════════════════════════════════════════════════════════════════════════


def make_embedder(*, model: str | None = None) -> Any:
    """Construct a Kaizen embedding adapter backed by Ollama.

    Args:
        model: Ollama embedding model tag. Defaults to
            :data:`DEFAULT_EMBED_MODEL`.

    Returns:
        An adapter with ``async def embed(texts: list[str]) -> list[list[float]]``.
    """
    from kaizen_agents.delegate.adapters.registry import get_embedding_adapter

    resolved = model or DEFAULT_EMBED_MODEL
    return get_embedding_adapter(
        "ollama",
        model=resolved,
        base_url=OLLAMA_BASE_URL,
    )


# ════════════════════════════════════════════════════════════════════════
# Streaming helpers — same call shape as the old `run_delegate`
# ════════════════════════════════════════════════════════════════════════


async def run_delegate_text(
    delegate: Any, prompt: str
) -> tuple[str, dict[str, int], float]:
    """Run a Delegate to completion, return ``(text, usage, latency_s)``.

    ``usage`` is a dict with ``prompt_tokens``, ``completion_tokens``,
    ``total_tokens`` (zero for events that don't carry usage). ``cost``
    is intentionally absent — Ollama is free, and Kaizen's ``consumed_usd``
    is mis-priced for local models.

    Event handling: the Kaizen ``TextDelta`` events expose the streamed
    chunk on ``.text`` (not ``.delta_text`` — that name is reserved for a
    different StreamEvent shape). The terminal ``TurnComplete`` event
    carries the final ``.text`` (entire response, not a delta) plus the
    ``.usage`` dict. We accumulate from ``TextDelta.text`` for streaming
    UX and fall back to ``TurnComplete.text`` if no deltas arrived.
    """
    t0 = time.perf_counter()
    response = ""
    usage: dict[str, int] = {
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0,
    }
    final_text: str | None = None
    async for event in delegate.run(prompt):
        ev_type = getattr(event, "event_type", None)
        text = getattr(event, "text", None)
        if ev_type == "text_delta" and text:
            response += text
        elif ev_type == "turn_complete":
            final_text = text or final_text
            ev_usage = getattr(event, "usage", None) or {}
            for k in usage:
                if k in ev_usage:
                    usage[k] = int(ev_usage[k])
    if not response and final_text:
        response = final_text
    elapsed = time.perf_counter() - t0
    return response, usage, elapsed


# ════════════════════════════════════════════════════════════════════════
# Colab installer — used by the generator's Cell 0
# ════════════════════════════════════════════════════════════════════════


def colab_install_and_start_ollama(*, log_path: str = "/tmp/ollama.log") -> None:
    """Install the Ollama binary and start the daemon (Colab only).

    Idempotent: skips install if the binary is already on PATH and
    skips daemon start if a process is already listening on
    :data:`OLLAMA_BASE_URL`. Raises :class:`OllamaUnreachableError`
    if either step fails.

    Not for local use — locally, students run ``ollama serve`` themselves.
    """
    if not _running_in_colab():
        raise RuntimeError(
            "colab_install_and_start_ollama() is Colab-only. "
            "On a local machine, run `ollama serve` in a terminal."
        )

    # Step 1 — install if missing
    if shutil.which("ollama") is None:
        print("[colab] installing ollama …", flush=True)
        proc = subprocess.run(
            "curl -fsSL https://ollama.com/install.sh | sh",
            shell=True,
            check=False,
            capture_output=True,
            text=True,
        )
        if proc.returncode != 0:
            raise OllamaUnreachableError(
                f"ollama install failed (exit {proc.returncode}):\n"
                f"stdout: {proc.stdout[-500:]}\nstderr: {proc.stderr[-500:]}"
            )

    # Step 2 — start daemon if not already listening
    try:
        httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=1.0).raise_for_status()
        print("[colab] ollama daemon already running", flush=True)
        return
    except (httpx.HTTPError, httpx.ConnectError):
        pass

    print(f"[colab] starting ollama daemon (logs → {log_path}) …", flush=True)
    subprocess.Popen(
        f"nohup ollama serve > {log_path} 2>&1 &",
        shell=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    # Step 3 — wait up to 30s for the daemon to answer /api/tags
    deadline = time.monotonic() + 30.0
    last_err: Exception | None = None
    while time.monotonic() < deadline:
        try:
            httpx.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=1.0).raise_for_status()
            print("[colab] ollama daemon ready", flush=True)
            return
        except (httpx.HTTPError, httpx.ConnectError) as exc:
            last_err = exc
            time.sleep(0.5)

    raise OllamaUnreachableError(
        f"ollama daemon failed to come up within 30s. "
        f"Last error: {type(last_err).__name__}: {last_err}. "
        f"Inspect {log_path} for details."
    )


# ── shared/mlfp06/ex_3.py ──
"""
Shared infrastructure for MLFP06 Exercise 3 — DPO Preference Alignment.

Contains: UltraFeedback loading, DPO loss helper, GRPO advantage helper,
visualisation utilities, output paths. Technique-specific code does NOT
belong here.
"""

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
import torch.nn.functional as F


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()


MODEL_NAME = DEFAULT_CHAT_MODEL
BASE_MODEL = os.environ.get("SFT_BASE_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")

# Output directories
OUTPUT_DIR = Path("outputs") / "ex3_dpo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CACHE = Path("data") / "mlfp06" / "ultrafeedback"
DATA_CACHE.mkdir(parents=True, exist_ok=True)
CACHE_FILE = DATA_CACHE / "ultrafeedback_2k.parquet"

ADAPTER_OUTPUT_DIR = Path("./dpo_output")


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — UltraFeedback Binarized preference pairs
# ════════════════════════════════════════════════════════════════════════


def load_ultrafeedback(n_samples: int = 2000) -> pl.DataFrame:
    """Load UltraFeedback Binarized preference dataset as a polars DataFrame.

    Columns: prompt, chosen, rejected. Cached as parquet after first load.
    """
    if CACHE_FILE.exists():
        print(f"Loading cached preference pairs from {CACHE_FILE}")
        return pl.read_parquet(CACHE_FILE)

    print("Downloading UltraFeedback Binarized from HuggingFace (first run)...")
    from datasets import load_dataset

    ds = load_dataset("trl-lib/ultrafeedback_binarized", split="train")
    ds = ds.shuffle(seed=42).select(range(min(n_samples, len(ds))))

    def _extract(row: dict) -> dict:
        chosen_msgs = row["chosen"]
        rejected_msgs = row["rejected"]
        prompt = ""
        for msg in chosen_msgs:
            if msg.get("role") == "user":
                prompt = msg.get("content", "")
                break
        chosen_text = next(
            (m["content"] for m in chosen_msgs if m.get("role") == "assistant"),
            "",
        )
        rejected_text = next(
            (m["content"] for m in rejected_msgs if m.get("role") == "assistant"),
            "",
        )
        return {
            "prompt": prompt,
            "chosen": chosen_text,
            "rejected": rejected_text,
        }

    rows = [_extract(r) for r in ds]
    rows = [r for r in rows if r["prompt"] and r["chosen"] and r["rejected"]]
    pref_data = pl.DataFrame(rows)
    pref_data.write_parquet(CACHE_FILE)
    print(f"Cached {pref_data.height} preference pairs to {CACHE_FILE}")
    return pref_data


def split_preferences(
    pref_data: pl.DataFrame, train_frac: float = 0.9
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Split preference pairs into train/eval."""
    n_train = int(pref_data.height * train_frac)
    return pref_data[:n_train], pref_data[n_train:]


# ════════════════════════════════════════════════════════════════════════
# DPO LOSS — canonical implementation (re-used across technique files)
# ════════════════════════════════════════════════════════════════════════


def dpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    ref_chosen_logps: torch.Tensor,
    ref_rejected_logps: torch.Tensor,
    beta: float = 0.1,
) -> torch.Tensor:
    """Compute the DPO loss from scratch.

    Args:
        policy_chosen_logps:   log P_policy(y_w | x)   [batch]
        policy_rejected_logps: log P_policy(y_l | x)   [batch]
        ref_chosen_logps:      log P_ref(y_w | x)      [batch]
        ref_rejected_logps:    log P_ref(y_l | x)      [batch]
        beta:                  Temperature controlling preference strength.

    Returns:
        Scalar DPO loss (averaged over batch).
    """
    chosen_log_ratio = policy_chosen_logps - ref_chosen_logps
    rejected_log_ratio = policy_rejected_logps - ref_rejected_logps
    logits = beta * (chosen_log_ratio - rejected_log_ratio)
    return -F.logsigmoid(logits).mean()


# ════════════════════════════════════════════════════════════════════════
# GRPO — group-relative advantage helper
# ════════════════════════════════════════════════════════════════════════


def grpo_advantages(rewards: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Compute GRPO group-relative advantages (Shao et al., 2024, DeepSeekMath).

        A_i = (r_i - mean(r_group)) / (std(r_group) + eps)

    rewards: [n_prompts, K] tensor of scalar rewards, K completions per prompt.
    Returns advantages of the same shape: each row has mean ~0 and std ~1,
    so the update size does not depend on the reward scale. A group whose
    completions all score the same gets advantage 0 (no learning signal).
    """
    group_mean = rewards.mean(dim=1, keepdim=True)
    group_std = rewards.std(dim=1, keepdim=True)
    return (rewards - group_mean) / (group_std + eps)


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION — "Seeing Is Believing" for alignment
# ════════════════════════════════════════════════════════════════════════


def show_preference_length_distribution(
    pref_data: pl.DataFrame, title: str = "Chosen vs Rejected Length"
) -> Path:
    """Histogram of chosen vs rejected response lengths (character count)."""
    chosen_lens = np.array([len(s) for s in pref_data["chosen"].to_list()])
    rejected_lens = np.array([len(s) for s in pref_data["rejected"].to_list()])

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(
        chosen_lens,
        bins=60,
        alpha=0.65,
        label=f"Chosen (median={int(np.median(chosen_lens))})",
        color="#2E7D32",
    )
    ax.hist(
        rejected_lens,
        bins=60,
        alpha=0.65,
        label=f"Rejected (median={int(np.median(rejected_lens))})",
        color="#C62828",
    )
    ax.set_xlabel("Response length (characters)")
    ax.set_ylabel("Count")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.legend()
    fname = OUTPUT_DIR / "ex3_preference_lengths.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")
    return fname


def show_dpo_loss_curve_by_margin(
    beta: float = 0.1, title: str = "DPO loss vs policy preference margin"
) -> Path:
    """Visual proof: as the policy's preference margin grows, DPO loss drops."""
    margins = torch.linspace(-3, 3, 121)
    losses = [-F.logsigmoid(torch.tensor(beta * m)).item() for m in margins]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(margins.numpy(), losses, color="#1565C0", linewidth=2.2)
    ax.axvline(0, color="#616161", linestyle=":", label="No preference (margin=0)")
    ax.set_xlabel("Policy log-ratio margin  (chosen - rejected)")
    ax.set_ylabel("DPO loss")
    ax.set_title(f"{title}  (beta={beta})", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fname = OUTPUT_DIR / "ex3_dpo_loss_curve.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")
    return fname


def show_beta_sensitivity(betas: list[float], losses: list[float]) -> Path:
    """Bar chart of DPO loss for a synthetic batch across beta values."""
    fig, ax = plt.subplots(figsize=(9, 5))
    colors = [
        "#90CAF9" if b <= 0.1 else "#FFB74D" if b <= 0.3 else "#E57373" for b in betas
    ]
    bars = ax.bar([f"{b}" for b in betas], losses, color=colors, edgecolor="black")
    for bar, loss_val in zip(bars, losses):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{loss_val:.3f}",
            ha="center",
            fontsize=9,
        )
    ax.set_xlabel("Beta (KL-penalty coefficient)")
    ax.set_ylabel("DPO loss (synthetic batch)")
    ax.set_title(
        "Beta sensitivity — same margin, higher beta = lower loss (less drift needed)",
        fontsize=13,
        fontweight="bold",
    )
    fname = OUTPUT_DIR / "ex3_beta_sensitivity.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")
    return fname


def show_grpo_advantages(rewards: torch.Tensor, advantages: torch.Tensor) -> Path:
    """Visualise GRPO advantages for each prompt group."""
    n_prompts, K = rewards.shape
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    im0 = axes[0].imshow(rewards.numpy(), aspect="auto", cmap="viridis")
    axes[0].set_title("Rewards  r(x, y_i)", fontsize=12, fontweight="bold")
    axes[0].set_xlabel(f"Completion index (K={K})")
    axes[0].set_ylabel("Prompt index")
    fig.colorbar(im0, ax=axes[0])

    im1 = axes[1].imshow(
        advantages.numpy(),
        aspect="auto",
        cmap="RdBu_r",
        vmin=-abs(advantages).max(),
        vmax=abs(advantages).max(),
    )
    axes[1].set_title(
        "GRPO advantages  (r_i - mean) / std", fontsize=12, fontweight="bold"
    )
    axes[1].set_xlabel(f"Completion index (K={K})")
    axes[1].set_ylabel("Prompt index")
    fig.colorbar(im1, ax=axes[1])

    fname = OUTPUT_DIR / "ex3_grpo_advantages.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")
    return fname


def show_safety_refusal_rates(
    base_rate: float, aligned_rate: float, n_prompts: int
) -> Path:
    """Bar chart comparing base-model vs DPO-aligned refusal rates."""
    fig, ax = plt.subplots(figsize=(8, 5))
    labels = ["Base (pre-DPO)", "DPO-aligned"]
    values = [base_rate * 100, aligned_rate * 100]
    colors = ["#9E9E9E", "#2E7D32"]
    bars = ax.bar(labels, values, color=colors, edgecolor="black")
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{val:.0f}%",
            ha="center",
            fontsize=12,
            fontweight="bold",
        )
    ax.set_ylim(0, 110)
    ax.set_ylabel("Refusal rate on adversarial prompts (%)")
    ax.set_title(
        f"DPO safety alignment — {n_prompts} adversarial prompts",
        fontsize=13,
        fontweight="bold",
    )
    fname = OUTPUT_DIR / "ex3_safety_refusal_rates.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")
    return fname




# ════════════════════════════════════════════════════════════════════════
# MLFP06 — Exercise 3.2: The DPO Loss From Scratch
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Derive DPO from the RLHF objective (bypass the reward model)
#   - Implement the DPO loss in PyTorch and verify against known cases
#   - Understand beta as the KL-penalty coefficient (the anchor to pi_ref)
#   - Visualise how the DPO loss responds to policy preference margin
#   - Run a beta sensitivity sweep and interpret the curve
#   - Apply to MAS-compliant model alignment in Singapore finance
#
# PREREQUISITES: 01_preference_data.py (preference triple intuition).
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Derivation walkthrough: RLHF -> Bradley-Terry -> closed-form DPO
#   2. Implement dpo_loss() and verify with synthetic log-probabilities
#   3. Visualise loss vs policy preference margin
#   4. Beta sensitivity sweep on a fixed batch
#   5. Apply: pick beta for a MAS-regulated Singapore robo-advisor
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import polars as pl
import torch



## THEORY — RLHF to DPO in one page


RLHF objective:
    max_pi E[r(x,y)] - beta * KL(pi || pi_ref)

  -> Optimal policy:
    pi*(y|x) proportional to  pi_ref(y|x) * exp(r(x,y) / beta)

  -> Plug into Bradley-Terry. Z(x) cancels.

  -> DPO loss:
    L = -E[log sigma(beta * (log pi(y_w|x)/pi_ref(y_w|x)
                            - log pi(y_l|x)/pi_ref(y_l|x)))]


In [ ]:
# RLHF objective (maximise reward, minimise KL from reference policy):
#   max_pi  E_x,y~pi [r(x,y)]  -  beta * KL(pi || pi_ref)
#
# Closed-form optimal policy:
#   pi*(y|x) = pi_ref(y|x) * exp(r(x,y) / beta) / Z(x)
#
# Bradley-Terry preference model:
#   P(y_w > y_l | x) = sigma(r(x,y_w) - r(x,y_l))
#
# Substitute pi* back into Bradley-Terry. The partition Z(x) cancels
# because it appears in both sides of the preference. What's left:
#   P(y_w > y_l | x) = sigma(
#       beta * ( log pi(y_w|x)/pi_ref(y_w|x)
#              - log pi(y_l|x)/pi_ref(y_l|x) )
#   )
#
# Negative log-likelihood of preferences = the DPO loss:
#   L_DPO = -E[ log sigma(beta * (log pi(y_w|x)/pi_ref(y_w|x)
#                                - log pi(y_l|x)/pi_ref(y_l|x))) ]
#
# KEY INSIGHT: the reward model is IMPLICIT in the policy. DPO bypasses
# reward-model training entirely — it directly optimises the policy to
# satisfy preferences. Cheaper, simpler, and empirically very strong.

print("=" * 70)
print("TASK 1: DPO derivation walkthrough")
print("=" * 70)
print(
)



## TASK 2 — Implement dpo_loss() (already in shared/mlfp06/ex_3.py)
         Verify its behaviour on synthetic log-probabilities


In [ ]:
print("=" * 70)
print("TASK 2: Verify DPO loss on synthetic log-probs")
print("=" * 70)

torch.manual_seed(42)
batch_size = 16

# Policy clearly prefers chosen (high log-prob) over rejected (low log-prob).
# We use a clear mean separation (+1.0 vs -1.0) with small noise (×0.3) so the
# demonstrated ordering (reversed prefs -> higher loss) is unambiguous and does
# NOT depend on the exact RNG stream — importing shared.mlfp06 advances the
# global torch RNG, so a subtle ±0.5 separation could flip the comparison.
policy_chosen = torch.randn(batch_size) * 0.3 + 1.0
policy_rejected = torch.randn(batch_size) * 0.3 - 1.0
ref_chosen = torch.randn(batch_size) * 0.3
ref_rejected = torch.randn(batch_size) * 0.3

loss_val = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=0.1)
print(f"DPO loss (synthetic batch, beta=0.1): {loss_val.item():.4f}")

# Perfect alignment — very high chosen log-prob, very low rejected log-prob
perfect_chosen = torch.zeros(batch_size)
perfect_rejected = torch.full((batch_size,), -10.0)
loss_perfect = dpo_loss(
    perfect_chosen, perfect_rejected, ref_chosen, ref_rejected, beta=0.1
)
print(f"DPO loss (perfect alignment):         {loss_perfect.item():.4f}")

# Reversed preferences — loss should spike
loss_reversed = dpo_loss(
    policy_rejected, policy_chosen, ref_chosen, ref_rejected, beta=0.1
)
print(f"DPO loss (reversed preferences):      {loss_reversed.item():.4f}")



### Checkpoint 2


In [ ]:
assert loss_val.item() > 0, "DPO loss should be positive"
assert loss_perfect.item() < loss_val.item(), "Perfect alignment -> lower loss"
assert loss_reversed.item() > loss_val.item(), "Reversed prefs -> higher loss"
print("✓ Checkpoint 2 passed — DPO loss behaviour verified\n")



## VISUALISE — DPO loss as a function of the policy preference margin


In [ ]:
# The "margin" is: (log pi(y_w)/pi_ref(y_w)) - (log pi(y_l)/pi_ref(y_l))
# DPO loss is monotonic in this margin — as the model learns to prefer
# chosen over rejected relative to reference, loss drops smoothly.

show_dpo_loss_curve_by_margin(beta=0.1)



### Checkpoint Visual


In [ ]:
assert (OUTPUT_DIR / "ex3_dpo_loss_curve.png").exists()
print("✓ Visual checkpoint passed — loss-vs-margin curve saved\n")



## TASK 4 — Beta sensitivity sweep


Reading the sweep: the batch's log-ratio margin is fixed, and DPO scores
  it as beta * margin. At beta={betas[-1]} the loss is {beta_losses[-1]:.3f};
  at beta={betas[0]} it is {beta_losses[0]:.3f} (log 2 = 0.693 is "no preference").
  A HIGH beta rewards even a small departure from pi_ref, so the optimiser
  has little reason to move far. A LOW beta leaves the loss near log 2
  until the policy moves its log-ratios a long way from pi_ref.

  Interpretation of beta (the KL-penalty coefficient, pi* ~ pi_ref * exp(r/beta)):
    0.01 - 0.05 : Weak KL anchor. The policy can drift far from the SFT
                  reference to fit the preference data. Risk: over-fitting
                  the preferences (verbosity, over-refusal, lost skills).
    0.1         : Standard default. Balanced for most tasks.
    0.2 - 0.5   : Strong KL anchor. Stays close to the SFT reference;
                  preferences nudge rather than rewrite behaviour.
    >= 1.0      : Very strong anchor. The policy barely moves — the
                  preference signal may not change behaviour at all.


In [ ]:
print("=" * 70)
print("TASK 4: Beta sensitivity sweep")
print("=" * 70)

betas = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
beta_losses = []
for b in betas:
    loss_b = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=b)
    beta_losses.append(loss_b.item())

beta_df = pl.DataFrame({"beta": betas, "dpo_loss": beta_losses})
print(beta_df)

show_beta_sensitivity(betas, beta_losses)

print(
)



### Checkpoint 4


In [ ]:
assert len(beta_losses) == len(betas)
assert (OUTPUT_DIR / "ex3_beta_sensitivity.png").exists()
print("✓ Checkpoint 4 passed — beta sensitivity sweep complete\n")



## APPLY — Pick beta for a MAS-regulated Singapore robo-advisor


In [ ]:
# BUSINESS SCENARIO: A Singapore robo-advisor must comply with MAS
# Notice FAA-N16 on investment advice. The model must refuse unlicensed
# advice, refuse to promise returns, and always disclose risk.
#
# DECISION: what beta should you pick for DPO? (beta is the KL anchor)
#   - Too low (0.02): weak anchor, the policy drifts far from the SFT
#     base to fit the refusal-heavy preference data -> it refuses benign
#     questions like "what is a stock?" -> user frustration, churn
#   - Too high (1.0): strong anchor, the policy barely moves from the
#     SFT base -> it still hedges on risk warnings inconsistently ->
#     compliance risk and possible regulatory action
#
# RULE OF THUMB for regulated finance: start near the default
# (beta=0.1-0.2), measure risk-disclosure compliance AND over-refusal on
# held-out eval sets, then tune: raise beta if benign questions start
# getting refused, lower it if disclosures are still inconsistent.
# All figures below are ILLUSTRATIVE planning numbers, not measurements.

print("=" * 70)
print("APPLICATION — Robo-advisor DPO beta selection")
print("=" * 70)

# Illustrative planning figures (not measurements)
AUM_SGD = 450_000_000  # Assets under management
EXPECTED_ENFORCEMENT_COST = 1_200_000  # Expected value of a single enforcement
AVG_ANNUAL_REVENUE_PER_USER = 180  # SGD per user per year
USERS_LOST_OVER_REFUSAL = 3500  # Churn from over-refusing benign queries

cost_low_beta = USERS_LOST_OVER_REFUSAL * AVG_ANNUAL_REVENUE_PER_USER
cost_high_beta = EXPECTED_ENFORCEMENT_COST
cost_balanced = 0.2 * cost_low_beta + 0.25 * cost_high_beta

rows = [
    {
        "beta": 0.02,
        "approx_annual_cost_sgd": cost_low_beta,
        "risk": "Over-refusal — weak KL anchor, churn from frustrated users",
    },
    {
        "beta": 0.20,
        "approx_annual_cost_sgd": cost_balanced,
        "risk": "Balanced — target for MAS-regulated deployments",
    },
    {
        "beta": 1.00,
        "approx_annual_cost_sgd": cost_high_beta,
        "risk": "Enforcement — policy barely moves, disclosures inconsistent",
    },
]
cost_df = pl.DataFrame(rows)
print(cost_df)

print(
    f"\nRecommended: beta = 0.20  "
    f"(estimated annual cost ~ S${cost_balanced:,.0f} vs worst-case "
    f"S${max(cost_low_beta, cost_high_beta):,.0f})"
)



### Checkpoint Application


In [ ]:
assert cost_df.height == 3
assert cost_balanced < max(cost_low_beta, cost_high_beta)
print("✓ Application checkpoint passed — beta selection justified\n")



## REFLECTION


[x] Derived DPO from RLHF + Bradley-Terry in one page
  [x] Implemented and verified the DPO loss with three sanity cases
      (typical, perfect alignment, reversed preferences)
  [x] Visualised the loss as a smooth function of preference margin
  [x] Swept beta from 0.01 -> 1.0 and interpreted the curve
  [x] Applied beta selection to a MAS-regulated robo-advisor — framed
      the trade-off as expected-cost in S$

  KEY INSIGHT: Beta is the KL-penalty coefficient. HIGH beta keeps the
  policy close to the SFT reference; LOW beta lets it drift far to fit
  the preferences (and over-fit them). For regulated deployments, start
  near 0.1-0.2, measure disclosure compliance and over-refusal, and
  raise beta if the policy drifts into refusing benign questions.

  Next: 03_dpo_training.py runs the full AlignmentPipeline on UltraFeedback
  and registers the resulting adapter.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

